## Web search

In [1]:
import os
import json
import time
import random
import re
import requests
import webbrowser
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from serpapi import GoogleSearch

# SerpAPI Configuration
GoogleSearch.SERP_API_KEY = "45c9f68dd058e937b082a171952a097f1fe9c85f90dbe5fb4ae60b031bd872a8"

# Paths Configuration
BASE_FOLDER = '/Users/shengfang/Desktop/TRI/test_FAPbI3'
PDF_FOLDER = os.path.join(BASE_FOLDER, 'pdfs')
os.makedirs(BASE_FOLDER, exist_ok=True)
os.makedirs(PDF_FOLDER, exist_ok=True)

In [1]:
# Search for Research Papers using SerpAPI
def search_papers(query, num_results=10):
    """Search for academic papers and extract links"""
    search = GoogleSearch({
        "q": query,
        "engine": "google_scholar",
        "api_key": GoogleSearch.SERP_API_KEY,
        "num": num_results,
    })
    
    result = search.get_dict()
    
    # Save search results
    json_file_path = os.path.join(BASE_FOLDER, f'{query.replace(" ", "_")}_search_results.json')
    with open(json_file_path, 'w') as json_file:
        json.dump(result, json_file, indent=4)
    
    # Extract links
    links = []
    for item in result.get('organic_results', []):
        link = item.get('link')
        if link:
            links.append(link)
    
    # Save links
    csv_file_path = os.path.join(BASE_FOLDER, f'{query.replace(" ", "_")}_links.csv')
    with open(csv_file_path, 'w') as csv_file:
        for link in links:
            csv_file.write(link + '\n')
    
    print(f" Search completed for: {query}")
    print(f" Found {len(links)} research paper links")
    print(f" Results saved to: {json_file_path}")
    print(f" Links saved to: {csv_file_path}")
    
    return links, result

# Using multiple search terms increases coverage and quality of results

SEARCH_QUERIES = [
    # Primary search with full chemical name
    "formamidinium lead iodide spin coating thin films",

    # Abbreviation search
    "FAPbI3 spin coating thin films",

    # Alternative with chemical formula and full name
    "FAPbI3 formamidinium lead iodide film fabrication",
    
    # Perovskite context (important for this material class)
    "formamidinium lead iodide perovskite spin coating",

]

# Execute multiple searches for comprehensive coverage
all_links = []
all_search_results = []

for i, query in enumerate(SEARCH_QUERIES, 1):
    print(f"\n{'='*60}")
    print(f"SEARCH {i}/{len(SEARCH_QUERIES)}: {query}")
    print(f"{'='*60}")
    
    links, search_results = search_papers(query, num_results=8) 
    all_links.extend(links)
    all_search_results.append({
        'query': query,
        'results': search_results,
        'links_found': len(links)
    })
    
    # Brief pause between searches to be respectful to the API
    if i < len(SEARCH_QUERIES):
        time.sleep(2)

# Remove duplicates while preserving order
seen = set()
unique_links = []
for link in all_links:
    if link not in seen:
        seen.add(link)
        unique_links.append(link)

links = unique_links

print(f"\n{'='*60}")
print(f"COMPREHENSIVE SEARCH SUMMARY")
print(f"{'='*60}")
print(f"Total searches performed: {len(SEARCH_QUERIES)}")
print(f"=Total links found: {len(all_links)}")
print(f"Unique links after deduplication: {len(links)}")
print(f"\nSearch breakdown:")
for result in all_search_results:
    print(f"  • '{result['query'][:50]}...': {result['links_found']} links")

# Save comprehensive results
comprehensive_results = {
    'search_strategy': 'Multi-query comprehensive search for FAPbI3',
    'chemical_names': [
        'FAPbI3 (abbreviation)',
        'Formamidinium Lead Iodide (IUPAC name)',
        'HC(NH2)2PbI3 (chemical formula)',
        'Formamidinium Lead Triiodide'
    ],
    'total_queries': len(SEARCH_QUERIES),
    'queries_used': SEARCH_QUERIES,
    'total_links_found': len(all_links),
    'unique_links': len(links),
    'individual_results': all_search_results
}

comprehensive_results_file = os.path.join(BASE_FOLDER, 'comprehensive_FAPbI3_search_results.json')
with open(comprehensive_results_file, 'w') as f:
    json.dump(comprehensive_results, f, indent=4)

print(f"Comprehensive results saved to: {comprehensive_results_file}")


print(f"\n Found Links:")
for i, link in enumerate(links, 1):
    print(f"  {i}. {link}")


SEARCH 1/4: formamidinium lead iodide spin coating thin films


NameError: name 'GoogleSearch' is not defined

In [4]:
# Enhanced Institutional PDF Downloader - Performance Optimized
class InstitutionalPDFDownloader:
    """
    Main PDF downloader with institutional access and anti-detection features
    
    Performance Optimizations:
    - Reduced timeouts from 15-30s to 8-15s
    - Removed duplicate Optica handling code
    - Limited PDF candidate attempts to first 3 per URL
    - Faster filename generation and error message truncation
    - Quick PDF validation without full content inspection
    """
    
    def __init__(self):
        self.session = requests.Session()
        self.setup_session()
        self.stats = {
            'downloaded': 0,
            'failed': 0,
            'semi_automated': 0
        }
    
    def setup_session(self):
        """Setup realistic browser session to avoid bot detection"""
        user_agents = [
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2.1 Safari/605.1.15'
        ]
        
        selected_ua = random.choice(user_agents)
        
        self.session.headers.update({
            'User-Agent': selected_ua,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate, br',
            'Connection': 'keep-alive',
            'Upgrade-Insecure-Requests': '1',
            'Sec-Fetch-Dest': 'document',
            'Sec-Fetch-Mode': 'navigate',
            'Sec-Fetch-Site': 'none',
            'Cache-Control': 'max-age=0'
        })
    
    def construct_pdf_urls(self, url):
        """Construct direct PDF URLs based on publisher patterns - Enhanced with robust fallbacks"""
        pdf_candidates = []
        domain = urlparse(url).netloc.lower()
        
        print(f"    Analyzing domain: {domain}")
        
        # ===== ENHANCED ARXIV HANDLING =====
        if 'arxiv.org' in domain:
            if '/abs/' in url:
                # Direct conversion from abs to pdf
                pdf_url = url.replace('/abs/', '/pdf/') + '.pdf'
                pdf_candidates.append(('constructed', pdf_url, 'arXiv PDF'))
                print(f"    ✅ Constructed arXiv PDF: {pdf_url}")
            else:
                # Fallback pattern for other arXiv URLs
                arxiv_match = re.search(r'arxiv\.org/.*?(\d+\.\d+)', url)
                if arxiv_match:
                    paper_id = arxiv_match.group(1)
                    pdf_url = f"https://arxiv.org/pdf/{paper_id}.pdf"
                    pdf_candidates.append(('constructed', pdf_url, f'arXiv PDF ({paper_id})'))
                    print(f"    ✅ Constructed arXiv PDF from ID: {pdf_url}")
        
        # ===== ENHANCED WILEY SUPPORT =====
        elif ('wiley.com' in domain or 'chemistry-europe' in domain) and '/doi/' in url:
            # Multiple Wiley patterns for better success rate
            base_url = url.split('/doi/')[0] if '/doi/' in url else url
            doi_part = url.split('/doi/')[1] if '/doi/' in url else ''
            
            wiley_patterns = []
            if '/doi/full/' in url:
                wiley_patterns.append(url.replace('/doi/full/', '/doi/pdf/'))
            elif '/doi/abs/' in url:
                wiley_patterns.append(url.replace('/doi/abs/', '/doi/pdf/'))
            elif doi_part:
                wiley_patterns.extend([
                    f"{base_url}/doi/pdf/{doi_part}",
                    f"{base_url}/doi/pdfdirect/{doi_part}"
                ])
            
            for pattern in wiley_patterns:
                pdf_candidates.append(('constructed', pattern, 'Wiley PDF'))
            print(f"    ✅ Constructed {len(wiley_patterns)} Wiley PDF patterns")
            
        # ===== ENHANCED ACS SUPPORT =====
        elif 'acs.org' in domain and '/doi/' in url:
            base_url = url.split('/doi/')[0]
            doi_part = url.split('/doi/')[1]
            acs_patterns = [
                f"{base_url}/doi/pdf/{doi_part}",
                f"{base_url}/doi/pdfplus/{doi_part}"
            ]
            for pattern in acs_patterns:
                pdf_candidates.append(('constructed', pattern, 'ACS PDF'))
            print(f"    ✅ Constructed {len(acs_patterns)} ACS PDF patterns")
            
        # ===== ENHANCED SCIENCEDIRECT SUPPORT =====
        elif 'sciencedirect.com' in domain:
            # Enhanced ScienceDirect: extract PII and try multiple patterns
            if '/pii/' in url:
                pii_match = re.search(r'/pii/([A-Z0-9]+)', url)
                if pii_match:
                    pii = pii_match.group(1)
                    # Multiple PDF URL patterns for better success rate
                    sciencedirect_patterns = [
                        f"https://www.sciencedirect.com/science/article/pii/{pii}/pdfft?md5=b&pid=1-s2.0-{pii}-main.pdf",
                        f"https://www.sciencedirect.com/science/article/pii/{pii}/pdf",
                        f"https://pdf.sciencedirectassets.com/science/article/pii/{pii}/1-s2.0-{pii}-main.pdf"
                    ]
                    for i, pattern in enumerate(sciencedirect_patterns, 1):
                        pdf_candidates.append(('constructed', pattern, f'ScienceDirect PDF Pattern {i} (PII: {pii})'))
                    print(f"    ✅ Constructed {len(sciencedirect_patterns)} ScienceDirect PDF patterns")
                    
        # ===== ENHANCED NATURE SUPPORT =====
        elif 'nature.com' in domain and '/articles/' in url:
            # Nature: add .pdf extension
            pdf_url = url.rstrip('/') + '.pdf'
            pdf_candidates.append(('constructed', pdf_url, 'Nature PDF'))
            print(f"    ✅ Constructed Nature PDF: {pdf_url}")
            
        # ===== ENHANCED PMC SUPPORT =====
        elif 'pmc.ncbi.nlm.nih.gov' in domain or 'pubmed' in domain:
            # PubMed Central - enhanced with multiple patterns
            try:
                # Try direct construction first (faster)
                pmc_match = re.search(r'PMC(\d+)', url)
                if pmc_match:
                    pmc_id = pmc_match.group(1)
                    pmc_patterns = [
                        f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/pdf/",
                        f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/pdf/main.pdf"
                    ]
                    for pattern in pmc_patterns:
                        pdf_candidates.append(('constructed', pattern, f'PMC PDF (PMC{pmc_id})'))
                    print(f"    ✅ Constructed {len(pmc_patterns)} PMC PDF patterns")
                else:
                    # Fallback to parsing if no PMC ID found
                    response = self.session.get(url, timeout=8)  # Reduced timeout
                    if response.status_code == 200:
                        soup = BeautifulSoup(response.content, 'html.parser')
                        for a_tag in soup.find_all('a', href=True):
                            href = a_tag.get('href', '')
                            text = a_tag.get_text(strip=True).lower()
                            if 'pdf' in text and ('download' in text or 'view' in text):
                                full_url = href if href.startswith('http') else urljoin(url, href)
                                pdf_candidates.append(('parsed', full_url, 'PMC PDF'))
                        print(f"    ✅ Found {len(pdf_candidates)} PMC PDF candidates")
            except Exception as e:
                print(f"    ⚠️ PMC failed: {str(e)[:30]}...")
                
        # ===== ENHANCED RSC SUPPORT =====
        elif 'rsc.org' in domain:
            # RSC: multiple pattern attempts
            if 'articlelanding' in url:
                # Direct pattern replacement
                pdf_url = url.replace('articlelanding', 'articlepdf')
                pdf_candidates.append(('constructed', pdf_url, 'RSC PDF'))
                print(f"    ✅ Constructed RSC PDF pattern")
                
            # Try HTML parsing for supplementary PDFs
            try:
                response = self.session.get(url, timeout=8)  # Reduced timeout
                if response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    for a_tag in soup.find_all('a', href=True):
                        href = a_tag.get('href', '')
                        if href and (href.lower().endswith('.pdf') or '/suppdata/' in href):
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, 'RSC PDF'))
                    print(f"    ✅ Found {len(pdf_candidates)} RSC PDF links")
            except Exception as e:
                print(f"    ⚠️ RSC parsing failed: {str(e)[:30]}...")
                
        # ===== ENHANCED SPRINGER SUPPORT =====
        elif 'springer' in domain and '/doi/' in url:
            base_url = url.split('/doi/')[0]
            doi_part = url.split('/doi/')[1]
            springer_patterns = [
                f"{base_url}/doi/pdf/{doi_part}",
                f"{base_url}/article/{doi_part}/pdf"
            ]
            for pattern in springer_patterns:
                pdf_candidates.append(('constructed', pattern, 'Springer PDF'))
            print(f"    ✅ Constructed {len(springer_patterns)} Springer PDF patterns")
            
        # ===== ENHANCED OPTICA SUPPORT =====
        elif 'optica.org' in domain or 'opg.optica.org' in domain:
            # Optica (OSA) journals - enhanced handling
            try:
                response = self.session.get(url, timeout=8)  # Reduced timeout
                print(f"     Optica response status: {response.status_code}")
                
                if response.status_code == 202:
                    print(f"    ⚠️ HTTP 202 (Accepted) - Bot detection, using manual patterns")
                    # Construct PDF URLs based on the URI pattern
                    if 'abstract.cfm' in url and 'uri=' in url:
                        uri_match = re.search(r'uri=([^&]+)', url)
                        if uri_match:
                            uri = uri_match.group(1)
                            optica_patterns = [
                                f"https://opg.optica.org/viewmedia.cfm?uri={uri}&seq=0",
                                f"https://opg.optica.org/DirectPDFAccess/{uri}.pdf"
                            ]
                            for pattern in optica_patterns:
                                pdf_candidates.append(('manual', pattern, f'Optica PDF ({uri})'))
                    pdf_candidates.append(('manual', url, 'Optica - Manual access (HTTP 202)'))
                    
                elif response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    # Look for PDF download links
                    for a_tag in soup.find_all('a', href=True):
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True).lower()
                        if (('pdf' in text and any(word in text for word in ['download', 'full', 'view'])) or
                            href.endswith('.pdf') or 'viewmedia.cfm' in href):
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, f'Optica PDF'))
                    
                    # Direct PDF construction from abstract URL
                    if 'abstract.cfm' in url and 'uri=' in url:
                        uri_match = re.search(r'uri=([^&]+)', url)
                        if uri_match:
                            uri = uri_match.group(1)
                            pdf_url = f"https://opg.optica.org/viewmedia.cfm?uri={uri}&seq=0"
                            pdf_candidates.append(('constructed', pdf_url, f'Optica PDF ({uri})'))
                    
                    print(f"    ✅ Found {len(pdf_candidates)} Optica PDF candidates")
                else:
                    print(f"    ⚠️ HTTP {response.status_code} - adding as manual")
                    pdf_candidates.append(('manual', url, f'Optica - Manual access (HTTP {response.status_code})'))
                    
            except Exception as e:
                print(f"    ⚠️ Optica failed: {str(e)[:30]}...")
                pdf_candidates.append(('manual', url, 'Optica - Manual access required'))
        
        # ===== ENHANCED GENERIC PATTERN HANDLING =====
        # If no specific patterns found, try enhanced generic parsing with fallback patterns
        if not pdf_candidates:
            try:
                print(f"    Trying enhanced generic PDF parsing...")
                response = self.session.get(url, timeout=8)  # Reduced timeout
                
                # Handle special status codes
                if response.status_code == 202:
                    print(f"    ⚠️ HTTP 202 - needs authentication")
                    pdf_candidates.append(('manual', url, 'Requires authentication (HTTP 202)'))
                elif response.status_code == 403:
                    print(f"    ⚠️ HTTP 403 - institutional access required")
                    pdf_candidates.append(('manual', url, 'Institutional access required (HTTP 403)'))
                elif response.status_code == 200:
                    soup = BeautifulSoup(response.content, 'html.parser')
                    
                    # Look for PDF links (limit to first 5 for performance)
                    pdf_links_found = 0
                    for a_tag in soup.find_all('a', href=True):
                        if pdf_links_found >= 5:  # Limit for performance
                            break
                            
                        href = a_tag.get('href', '')
                        text = a_tag.get_text(strip=True).lower()
                        
                        # Check for PDF indicators
                        if (href.lower().endswith('.pdf') or 
                            'pdf' in text and any(word in text for word in ['download', 'full', 'view']) or
                            '/pdf' in href.lower()):
                            
                            full_url = href if href.startswith('http') else urljoin(url, href)
                            pdf_candidates.append(('parsed', full_url, 'Generic PDF'))
                            pdf_links_found += 1
                    
                    # ===== ENHANCED FALLBACK PATTERN CONSTRUCTION =====
                    # Try enhanced pattern construction for missed cases
                    if not pdf_candidates:
                        # DOI-based fallback patterns
                        if '/doi/' in url:
                            base_url = url.split('/doi/')[0]
                            doi_part = url.split('/doi/')[1]
                            fallback_patterns = [
                                f"{base_url}/doi/pdf/{doi_part}",
                                f"{base_url}/doi/pdfplus/{doi_part}",
                                f"{base_url}/doi/pdfdirect/{doi_part}"
                            ]
                            for pattern in fallback_patterns:
                                pdf_candidates.append(('constructed', pattern, 'DOI Fallback PDF'))
                            print(f"    ✅ Added {len(fallback_patterns)} DOI fallback patterns")
                        
                        # ArXiv fallback for missed cases
                        elif 'arxiv' in domain:
                            arxiv_match = re.search(r'(\d+\.\d+)', url)
                            if arxiv_match:
                                paper_id = arxiv_match.group(1)
                                pdf_url = f"https://arxiv.org/pdf/{paper_id}.pdf"
                                pdf_candidates.append(('constructed', pdf_url, f'arXiv Fallback ({paper_id})'))
                                print(f"    ✅ Added arXiv fallback pattern")
                    
                    if pdf_candidates:
                        print(f"    ✅ Found {len(pdf_candidates)} enhanced PDF candidates")
                    else:
                        print(f"     No PDF links found in HTML")
                        # Add the original URL as a manual candidate
                        pdf_candidates.append(('manual', url, 'Manual download required'))
                else:
                    print(f" HTTP {response.status_code} - adding as manual download")
                    pdf_candidates.append(('manual', url, f'Manual download required (HTTP {response.status_code})'))
                        
            except Exception as e:
                print(f"Enhanced parsing failed: {str(e)}")
                # Add as manual download candidate
                pdf_candidates.append(('manual', url, f'Manual download required (parsing failed)'))
        
        return pdf_candidates
    
    def download_pdf(self, pdf_url, save_path, original_url=None):
        """Download PDF with institutional access support - optimized for speed"""
        try:
            print(f" Downloading: {pdf_url[:60]}...")
            
            # Set proper referrer if provided
            if original_url:
                self.session.headers.update({'Referer': original_url})
            
            response = self.session.get(pdf_url, timeout=15, allow_redirects=True)  # Reduced timeout
            
            print(f" Status: {response.status_code}")
            
            if response.status_code == 200:
                content = response.content
                
                # Quick PDF validation
                if len(content) > 1000 and content.startswith(b'%PDF-'):
                    with open(save_path, 'wb') as f:
                        f.write(content)
                    file_size = os.path.getsize(save_path)
                    self.stats['downloaded'] += 1
                    return True, f"Downloaded ({file_size//1024} KB)"
                
                # Check content type
                content_type = response.headers.get('Content-Type', '').lower()
                if 'application/pdf' in content_type and len(content) > 1000:
                    with open(save_path, 'wb') as f:
                        f.write(content)
                    file_size = os.path.getsize(save_path)
                    self.stats['downloaded'] += 1
                    return True, f"Downloaded ({file_size//1024} KB)"
                
                # Quick HTML check
                if b'<html' in content[:500].lower():
                    return False, "Institutional auth required (HTML page)"
                else:
                    return False, "Not a valid PDF"
            
            elif response.status_code == 403:
                return False, "Access forbidden - institutional auth required"
            else:
                return False, f"HTTP {response.status_code}"
                
        except Exception as e:
            return False, str(e)[:30] + "..."
    
    def open_in_browser(self, pdf_url, title="PDF"):
        """Open PDF URL in browser for semi-automated download"""
        try:
            print(f"Opening {title} in browser...")
            webbrowser.open(pdf_url)
            self.stats['semi_automated'] += 1
            return True, "Opened in browser"
        except Exception as e:
            return False, str(e)

print("✅ InstitutionalPDFDownloader class created")

✅ InstitutionalPDFDownloader class created


In [ ]:
def download_pdfs_from_links(links, pdf_folder):
    """
    Enhanced main function to download PDFs from research links with intelligent fallback
    Incorporates robust patterns and better error handling
    
    Returns:
        tuple: (downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs)
    """
    downloader = InstitutionalPDFDownloader()
    
    downloaded_pdfs = []
    failed_downloads = []
    semi_automated_pdfs = []
    
    print(f"🚀 STARTING ENHANCED PDF DOWNLOAD PROCESS")
    print("=" * 60)
    
    for i, link in enumerate(links, 1):
        print(f"\n[{i}/{len(links)}] Processing: {link}")
        
        try:
            # Get PDF candidates using enhanced publisher-specific patterns
            pdf_candidates = downloader.construct_pdf_urls(link)
            
            if not pdf_candidates:
                print(f"    ⚠️  No PDF candidates found")
                failed_downloads.append({"url": link, "error": "No PDF candidates found", "category": "no_candidates"})
                continue

            print(f"    Found {len(pdf_candidates)} PDF candidates")

            # ===== ENHANCED CANDIDATE PRIORITIZATION =====
            # Sort candidates by priority: constructed > parsed > manual
            priority_order = {'constructed': 1, 'parsed': 2, 'manual': 3}
            pdf_candidates.sort(key=lambda x: priority_order.get(x[0], 4))
            
            # Try downloading each PDF candidate with intelligent limits
            pdf_downloaded = False
            best_manual_candidate = None
            download_attempts = 0
            max_attempts = min(5, len(pdf_candidates))  # Try up to 5 candidates
            
            for method, pdf_url, description in pdf_candidates[:max_attempts]:
                if pdf_downloaded:
                    break
                
                download_attempts += 1
                print(f"    [{download_attempts}/{max_attempts}] Trying: {description}")
                
                # Generate clean filename with better naming
                parsed = urlparse(link)
                domain_clean = parsed.netloc.replace('www.', '').replace('.', '_')
                path_clean = parsed.path.replace('/', '_').strip('_')[:40]
                pdf_name = f"{domain_clean}_{path_clean}_{i}.pdf"
                pdf_path = os.path.join(pdf_folder, pdf_name)
                
                if method == 'manual':
                    # Add to semi-automated list (requires browser/manual access)
                    if not best_manual_candidate:
                        best_manual_candidate = {
                            "title": description,
                            "url": link,
                            "pdf_url": pdf_url,
                            "reason": "Requires institutional authentication or manual access",
                            "domain": parsed.netloc,
                            "priority": "high" if 'arxiv' in pdf_url.lower() or 'pmc' in pdf_url.lower() else "medium"
                        }
                    print(f"        🔖 Marked for browser opening: {description}")
                else:
                    # Attempt automatic download
                    success, message = downloader.download_pdf(pdf_url, pdf_path, link)
                    
                    if success:
                        print(f"        ✅ Downloaded: {pdf_name}")
                        downloaded_pdfs.append({
                            "url": link, 
                            "pdf_path": pdf_path, 
                            "pdf_name": pdf_name,
                            "method": description,
                            "domain": parsed.netloc
                        })
                        pdf_downloaded = True
                    else:
                        print(f"        ❌ Failed: {message[:50]}...")
                        
                        # ===== ENHANCED ERROR CATEGORIZATION =====
                        # Categorize errors for better fallback decisions
                        error_lower = message.lower()
                        if any(keyword in error_lower for keyword in 
                               ["institutional", "authentication", "forbidden", "403"]):
                            # This suggests institutional access is needed
                            if not best_manual_candidate:
                                best_manual_candidate = {
                                    "title": description,
                                    "url": link,
                                    "pdf_url": pdf_url,
                                    "reason": f"Institutional access required: {message}",
                                    "domain": parsed.netloc,
                                    "priority": "high"
                                }
                        elif any(keyword in error_lower for keyword in 
                                ["html", "not a valid pdf", "too small"]):
                            # This suggests we got a login/paywall page
                            if not best_manual_candidate:
                                best_manual_candidate = {
                                    "title": description,
                                    "url": link,
                                    "pdf_url": pdf_url,
                                    "reason": f"Paywall detected: {message}",
                                    "domain": parsed.netloc,
                                    "priority": "high"
                                }
            
            # ===== ENHANCED RESULT HANDLING =====
            # If no automatic download succeeded, handle intelligently
            if not pdf_downloaded:
                if best_manual_candidate:
                    print(f"    🌐 Adding to semi-automated: {best_manual_candidate['title'][:40]}...")
                    semi_automated_pdfs.append(best_manual_candidate)
                else:
                    error_info = {
                        "url": link, 
                        "error": f"All {download_attempts} download attempts failed",
                        "category": "all_failed",
                        "domain": urlparse(link).netloc,
                        "attempts": download_attempts
                    }
                    failed_downloads.append(error_info)
                    print(f"    ❌ Complete failure after {download_attempts} attempts")
            
            # Brief pause for rate limiting
            time.sleep(0.3)  # Slightly longer pause for stability
            
        except Exception as e:
            print(f"    ❌ Error processing {link}: {str(e)[:50]}...")
            failed_downloads.append({
                "url": link, 
                "error": str(e),
                "category": "processing_error",
                "domain": urlparse(link).netloc if link else "unknown"
            })
    
    # ===== ENHANCED SUMMARY =====
    print(f"\n{'='*60}")
    print(f"📊 ENHANCED DOWNLOAD SUMMARY")
    print(f"✅ Automatically downloaded: {len(downloaded_pdfs)} PDFs")
    print(f"🌐 Semi-automated (browser): {len(semi_automated_pdfs)} PDFs")
    print(f"❌ Failed: {len(failed_downloads)} PDFs")
    
    if downloaded_pdfs:
        print(f"\n✅ SUCCESSFUL DOWNLOADS:")
        for pdf in downloaded_pdfs:
            print(f"   • {pdf['pdf_name']} ({pdf['method']})")
    
    if semi_automated_pdfs:
        print(f"\n🌐 SEMI-AUTOMATED (Manual Required):")
        high_priority = [p for p in semi_automated_pdfs if p.get('priority') == 'high']
        medium_priority = [p for p in semi_automated_pdfs if p.get('priority') != 'high']
        
        if high_priority:
            print(f"   🔴 HIGH PRIORITY ({len(high_priority)}):")
            for pdf in high_priority:
                print(f"      • {pdf['domain']} - {pdf['reason'][:50]}...")
        
        if medium_priority:
            print(f"   🟡 MEDIUM PRIORITY ({len(medium_priority)}):")
            for pdf in medium_priority:
                print(f"      • {pdf['domain']} - {pdf['reason'][:50]}...")
    
    print(f"\n📁 Saved to: {pdf_folder}")
    
    return downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs

# Execute the enhanced download process with robust error handling
print("🚀 Starting enhanced PDF download process with robust patterns...")
downloader, downloaded_pdfs, failed_downloads, semi_automated_pdfs = download_pdfs_from_links(links, PDF_FOLDER)

# Validate the collection
validation_results = validate_pdf_collection(PDF_FOLDER, downloaded_pdfs, semi_automated_pdfs)

Starting optimized PDF download process...
STARTING PDF DOWNLOAD PROCESS

[1/23] Processing: https://link.springer.com/article/10.1007/s12034-016-1282-z
    Analyzing domain: link.springer.com
    Trying generic PDF parsing...
    ✅ Found 2 generic PDF links
    Found 2 PDF candidates
 Downloading: https://link.springer.com/content/pdf/10.1007/s12034-016-128...
    ✅ Found 2 generic PDF links
    Found 2 PDF candidates
 Downloading: https://link.springer.com/content/pdf/10.1007/s12034-016-128...
 Status: 200
    ✅ Downloaded: link.springer.com_article_10.1007_s12034-016-1282-z.pdf
 Status: 200
    ✅ Downloaded: link.springer.com_article_10.1007_s12034-016-1282-z.pdf

[2/23] Processing: https://www.nature.com/articles/s41598-020-79291-1
    Analyzing domain: www.nature.com
    ✅ Constructed Nature PDF: https://www.nature.com/articles/s41598-020-79291-1.pdf
    Found 1 PDF candidates
 Downloading: https://www.nature.com/articles/s41598-020-79291-1.pdf...

[2/23] Processing: https://www.n

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


     No PDF links found in HTML
    Found 1 PDF candidates
   Marked for browser opening
    Adding to semi-automated: Manual download required...

[17/23] Processing: https://onlinelibrary.wiley.com/doi/abs/10.1002/smtd.201700387
    Analyzing domain: onlinelibrary.wiley.com
    ✅ Constructed Wiley PDF: https://onlinelibrary.wiley.com/doi/pdf/10.1002/smtd.201700387
    Found 1 PDF candidates
 Downloading: https://onlinelibrary.wiley.com/doi/pdf/10.1002/smtd.2017003...

[17/23] Processing: https://onlinelibrary.wiley.com/doi/abs/10.1002/smtd.201700387
    Analyzing domain: onlinelibrary.wiley.com
    ✅ Constructed Wiley PDF: https://onlinelibrary.wiley.com/doi/pdf/10.1002/smtd.201700387
    Found 1 PDF candidates
 Downloading: https://onlinelibrary.wiley.com/doi/pdf/10.1002/smtd.2017003...
 Status: 200
    ❌ Failed: Not a valid PDF...
    Adding to semi-automated: Wiley PDF from abs DOI...
 Status: 200
    ❌ Failed: Not a valid PDF...
    Adding to semi-automated: Wiley PDF from abs DO

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


     No PDF links found in HTML
    Found 1 PDF candidates
   Marked for browser opening
    Adding to semi-automated: Manual download required...

[20/23] Processing: https://www.nature.com/articles/s41467-018-04029-7
    Analyzing domain: www.nature.com
    ✅ Constructed Nature PDF: https://www.nature.com/articles/s41467-018-04029-7.pdf
    Found 1 PDF candidates
 Downloading: https://www.nature.com/articles/s41467-018-04029-7.pdf...

[20/23] Processing: https://www.nature.com/articles/s41467-018-04029-7
    Analyzing domain: www.nature.com
    ✅ Constructed Nature PDF: https://www.nature.com/articles/s41467-018-04029-7.pdf
    Found 1 PDF candidates
 Downloading: https://www.nature.com/articles/s41467-018-04029-7.pdf...
 Status: 200
    ✅ Downloaded: nature.com_articles_s41467-018-04029-7.pdf
 Status: 200
    ✅ Downloaded: nature.com_articles_s41467-018-04029-7.pdf

[21/23] Processing: https://pubs.acs.org/doi/abs/10.1021/acsaem.8b01964
    Analyzing domain: pubs.acs.org
    ✅ Const

In [ ]:
# Enhanced Semi-Automated Download (Browser Opening)
def open_pdfs_in_browser(semi_automated_pdfs, downloader):
    """
    Enhanced function to open PDFs in browser for semi-automated download
    Prioritizes high-priority PDFs and provides better user guidance
    
    Args:
        semi_automated_pdfs: List of PDF info dictionaries
        downloader: Instance of InstitutionalPDFDownloader
    
    Returns:
        int: Number of PDFs successfully opened in browser
    """
    if not semi_automated_pdfs:
        print("🎉 No semi-automated downloads needed - all PDFs were fully automated!")
        return 0
    
    print(f"\n🌐 ENHANCED SEMI-AUTOMATED DOWNLOAD PROCESS")
    print("=" * 60)
    
    # Sort by priority (high priority first)
    high_priority = [p for p in semi_automated_pdfs if p.get('priority') == 'high']
    medium_priority = [p for p in semi_automated_pdfs if p.get('priority') != 'high']
    sorted_pdfs = high_priority + medium_priority
    
    opened_count = 0
    
    # Process high priority first
    if high_priority:
        print(f"\n🔴 HIGH PRIORITY PDFs ({len(high_priority)}) - Likely to be freely accessible:")
        for i, pdf_info in enumerate(high_priority, 1):
            print(f"\n[HIGH {i}/{len(high_priority)}] Opening: {pdf_info['title']}")
            print(f"    🌐 URL: {pdf_info['pdf_url']}")
            print(f"    📝 Domain: {pdf_info['domain']}")
            print(f"    💡 Note: {pdf_info['reason']}")
            
            # Provide specific guidance based on domain
            domain = pdf_info['domain'].lower()
            if 'arxiv' in domain:
                print(f"    ✅ arXiv - Should be freely downloadable!")
            elif 'pmc.ncbi.nlm.nih.gov' in domain:
                print(f"    ✅ PubMed Central - Open access repository")
            elif 'nature.com' in domain or 'science.org' in domain:
                print(f"    🔒 High-impact journal - May require institutional access")
            
            success, message = downloader.open_in_browser(pdf_info['pdf_url'], pdf_info['title'])
            
            if success:
                print(f"    ✅ Successfully opened in browser")
                opened_count += 1
            else:
                print(f"    ❌ Failed to open: {message}")
            
            # Brief pause between high priority opens
            if i < len(high_priority):
                print(f"    ⏳ Waiting 2 seconds before next high-priority PDF...")
                time.sleep(2)
    
    # Process medium priority
    if medium_priority:
        print(f"\n🟡 MEDIUM PRIORITY PDFs ({len(medium_priority)}) - May require institutional access:")
        for i, pdf_info in enumerate(medium_priority, 1):
            print(f"\n[MED {i}/{len(medium_priority)}] Opening: {pdf_info['title']}")
            print(f"    🌐 URL: {pdf_info['pdf_url']}")
            print(f"    📝 Domain: {pdf_info['domain']}")
            print(f"    💡 Note: {pdf_info['reason']}")
            
            # Provide domain-specific guidance
            domain = pdf_info['domain'].lower()
            if 'wiley' in domain or 'springer' in domain or 'elsevier' in domain:
                print(f"    🔒 Major publisher - Likely requires subscription")
            elif 'acs.org' in domain:
                print(f"    🔒 ACS journal - May have institutional access")
            elif 'rsc.org' in domain:
                print(f"    🔒 RSC journal - Check for open access version")
            
            success, message = downloader.open_in_browser(pdf_info['pdf_url'], pdf_info['title'])
            
            if success:
                print(f"    ✅ Successfully opened in browser")
                opened_count += 1
            else:
                print(f"    ❌ Failed to open: {message}")
            
            # Longer pause between medium priority opens
            if i < len(medium_priority):
                print(f"    ⏳ Waiting 3 seconds before next medium-priority PDF...")
                time.sleep(3)
    
    # Enhanced summary with actionable guidance
    print(f"\n📊 SEMI-AUTOMATED SUMMARY:")
    print(f"    ✅ Successfully opened in browser: {opened_count} PDFs")
    print(f"    🔴 High priority (likely free): {len(high_priority)} PDFs")
    print(f"    🟡 Medium priority (may need access): {len(medium_priority)} PDFs")
    
    if opened_count > 0:
        print(f"\n💡 DOWNLOAD GUIDANCE:")
        print(f"    1. Check each opened browser tab")
        print(f"    2. Look for 'Download PDF' or 'Full Text' buttons")
        print(f"    3. For paywalled content, try:")
        print(f"       • University VPN or campus network")
        print(f"       • Institutional library access")
        print(f"       • Open access versions on arXiv/PMC")
        print(f"       • Author's personal/institutional webpage")
        print(f"    4. Save PDFs to: {PDF_FOLDER}")
    
    return opened_count

# Execute enhanced semi-automated download if needed
if semi_automated_pdfs:
    print(f"\n" + "="*60)
    opened_count = open_pdfs_in_browser(semi_automated_pdfs, downloader)
    
    # Update validation after browser opening
    print(f"\n🔄 After browser opening, you can re-run validation:")
    print("validation_results = validate_pdf_collection(PDF_FOLDER, downloaded_pdfs, semi_automated_pdfs)")
else:
    print("\n🎉 All PDFs were downloaded automatically! No manual steps needed.")



 SEMI-AUTOMATED DOWNLOAD PROCESS

[1/14] Opening: Wiley PDF from abs DOI
    URL: https://advanced.onlinelibrary.wiley.com/doi/pdf/10.1002/adfm.201600715
    Note: Not a valid PDF
Opening Wiley PDF from abs DOI in browser...
    ✅ Successfully opened in browser
    Waiting 3 seconds before next PDF...
    ✅ Successfully opened in browser
    Waiting 3 seconds before next PDF...

[2/14] Opening: Institutional access required (HTTP 403)
    URL: https://www.mdpi.com/2673-4591/12/1/1
    Note: Requires institutional authentication
Opening Institutional access required (HTTP 403) in browser...
    ✅ Successfully opened in browser
    Waiting 3 seconds before next PDF...

[2/14] Opening: Institutional access required (HTTP 403)
    URL: https://www.mdpi.com/2673-4591/12/1/1
    Note: Requires institutional authentication
Opening Institutional access required (HTTP 403) in browser...
    ✅ Successfully opened in browser
    Waiting 3 seconds before next PDF...

[3/14] Opening: Institutiona

In [8]:
from paperqa import Settings, ask
from paperqa.settings import AgentSettings, ParsingSettings
import os
import asyncio
import glob

# Configure environment for local Ollama model usage
os.environ['OPENAI_API_KEY'] = "ollama"

# Local LLM configuration
local_llm_config = {
    'model_list': [{
        'model_name': 'ollama/llama3.2',
        'litellm_params': {
            'model': 'ollama/llama3.2',
            'api_base': "http://localhost:11434",
            'temperature': 0.1,  
            'max_tokens': 4096,  
        }
    }]
}

# Disable online API access for fully local operation
os.environ['CROSSREF_MAILTO'] = ""  # Disable Crossref metadata lookup
os.environ['SEMANTIC_SCHOLAR_API_KEY'] = ""  # Disable Semantic Scholar API
os.environ['PAPERPILE_API_KEY'] = ""  # Disable any other potential APIs

# PaperQA settings for LOCAL PDF analysis 
synthesis_settings = Settings(
    llm='ollama/llama3.2',
    llm_config=local_llm_config,
    summary_llm='ollama/llama3.2',
    summary_llm_config=local_llm_config,
    embedding='ollama/mxbai-embed-large',
    agent=AgentSettings(
        agent_llm='ollama/llama3.2', 
        agent_llm_config=local_llm_config,
        timeout=600,
        tool_names=["gen_answer", "gather_evidence"],  # Local tools only
    ),
    # NOTE: Update this path for your target composition
    paper_directory="/Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs",
    parsing=ParsingSettings(
        use_doc_details=False,  # Disable metadata enrichment
        chunk_size=3000,  
        overlap=300,
    ),
    agent_type="ToolSelector",  # Force local-only analysis
)

print("PaperQA configuration complete!")
print(f"PDF directory: {synthesis_settings.paper_directory}")

# Load local PDF documents
pdf_files = glob.glob(os.path.join(synthesis_settings.paper_directory, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files for analysis")

if pdf_files:
    print("\nAvailable research papers:")
    for i, pdf in enumerate(pdf_files[:5], 1):
        filename = os.path.basename(pdf)
        print(f"  {i}. {filename}")
    if len(pdf_files) > 5:
        print(f"  ... and {len(pdf_files) - 5} additional files")
else:
    print("No PDF files found in the specified directory")

PaperQA configuration complete!
PDF directory: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Found 23 PDF files for analysis

Available research papers:
  1. pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00170f.pdf
  2. pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10267h.pdf
  3. document.pdf
  4. science.aan2301.pdf
  5. materials-16-01049-v3.pdf
  ... and 18 additional files


In [9]:
import asyncio
import nest_asyncio
import os
import glob
from paperqa import Docs, Settings
from paperqa.settings import ParsingSettings

# Enable async support in Jupyter
nest_asyncio.apply()
os.environ['OPENAI_API_KEY'] = "ollama"

async def local_synthesis_analysis(query, pdf_directory, analysis_name):
    """
    Perform LOCAL-ONLY PaperQA analysis for synthesis research
    
    Args:
        query: Research question to analyze (customize for your target composition)
        pdf_directory: Path to PDF collection (update for different compositions)
        analysis_name: Description for progress tracking
    
    Returns:
        PaperQA result object with answer and contexts
    """
    print(f"Starting: {analysis_name}")
    print(f"Analyzing papers in: {pdf_directory}")
    
    # Fully offline settings - no metadata enrichment
    offline_settings = Settings(
        llm='ollama/llama3.2',
        llm_config={
            'model_list': [{
                'model_name': 'ollama/llama3.2',
                'litellm_params': {
                    'model': 'ollama/llama3.2',
                    'api_base': 'http://localhost:11434',
                    'temperature': 0.1,
                    'max_tokens': 2048,
                }
            }]
        },
        summary_llm='ollama/llama3.2',
        embedding='ollama/mxbai-embed-large',
        parsing=ParsingSettings(
            use_doc_details=False,  # Critical: Disable metadata lookup
            chunk_size=2000,
            overlap=200,
        )
    )
    
    # Initialize document collection
    docs = Docs()
    pdf_files = glob.glob(os.path.join(pdf_directory, "*.pdf"))
    
    if not pdf_files:
        print(" No PDF files found")
        return None
        
    print(f"Processing {len(pdf_files)} research papers...")
    
    # Load and process PDFs with offline settings
    for i, pdf_file in enumerate(pdf_files, 1):
        filename = os.path.basename(pdf_file)
        print(f"  {i}/{len(pdf_files)} Loading: {filename[:50]}...")
        await docs.aadd(pdf_file, settings=offline_settings)
    
    # Execute analysis query
    print("Analyzing with local LLM...")
    result = await docs.aquery(query, settings=offline_settings)
    print(f"{analysis_name} completed!")
    
    return result

def run_local_analysis(query, pdf_directory, analysis_name):
    """Synchronous wrapper for modular local analysis"""
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
    
    return loop.run_until_complete(local_synthesis_analysis(query, pdf_directory, analysis_name))


In [10]:
TARGET_COMPOSITION = "FAPbI3"  # Change this for your target material
pdf_dir = "/Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs"  # Update path as needed

# Focused query for direct synthesis information
direct_synthesis_query = f"""
Extract specific synthesis details for {TARGET_COMPOSITION} thin films:

PRECURSORS:
- Chemicals used (primary precursors and alternatives)
- Molar ratios and concentrations
- Purity requirements and suppliers
- Primary solvents (DMF, DMSO, others)
- Solvent ratios for mixed systems
- Dissolution conditions (temperature, time)

PROCESSING PARAMETERS:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

FILM PROPERTIES:
- Film uniformity (homogeneity, defects)
- Film morphology (crystallinity, grain size)
- Film thicknesses
- Structural and optical properties

Provide quantitative values when available.
"""

# Execute extraction
focused_result = run_local_analysis(direct_synthesis_query, pdf_dir, f"Direct {TARGET_COMPOSITION} Synthesis")

# Display results
if focused_result:
    print("\n" + "="*60)
    print(f"DIRECT {TARGET_COMPOSITION} SYNTHESIS RESULTS")
    print("="*60)
    
    # Extract answer
    if hasattr(focused_result, 'answer'):
        answer_text = focused_result.answer
    elif hasattr(focused_result, 'formatted_answer'):
        answer_text = focused_result.formatted_answer
    else:
        answer_text = str(focused_result)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(focused_result, 'contexts') and focused_result.contexts:
        print(f"\nEvidence: {len(focused_result.contexts)} relevant passages found")
        
        # Display example context
        print("\nExample supporting evidence:")
        try:
            first_ctx = focused_result.contexts[0]
            if hasattr(first_ctx, 'text'):
                ctx_text = str(first_ctx.text)[:200] if first_ctx.text else "No text"
            else:
                ctx_text = str(first_ctx)[:200]
            print(f"   {ctx_text}...")
        except:
            print("   [Evidence available but not displayable]")
    else:
        print("\nNo direct evidence found")

    print(f"\n✅ Direct {TARGET_COMPOSITION} synthesis analysis complete!")
else:
    print(f"❌ Direct {TARGET_COMPOSITION} synthesis analysis failed")

Starting: Direct FAPbI3 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Processing 23 research papers...
  1/23 Loading: pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00...
  2/23 Loading: pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10...
  2/23 Loading: pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10...
  3/23 Loading: document.pdf...
  3/23 Loading: document.pdf...
  4/23 Loading: science.aan2301.pdf...
  4/23 Loading: science.aan2301.pdf...
  5/23 Loading: materials-16-01049-v3.pdf...
  5/23 Loading: materials-16-01049-v3.pdf...
  6/23 Loading: nature.com_articles_s41467-018-04029-7.pdf...
  6/23 Loading: nature.com_articles_s41467-018-04029-7.pdf...
  7/23 Loading: 1-s2.0-S0038092X18309617-main.pdf...
  7/23 Loading: 1-s2.0-S0038092X18309617-main.pdf...
  8/23 Loading: nature.com_articles_s41598-020-79291-1.pdf...
  8/23 Loading: nature.com_articles_s41598-020-79291-1.pdf...
  9/23 Loading: 1-s2.0-S2211285524010218-main.pdf...
  9/23 Loading: 1

Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.


Direct FAPbI3 Synthesis completed!

DIRECT FAPbI3 SYNTHESIS RESULTS
The synthesis details for FAPbI3 thin films are as follows:

**Precurors:**

* Primary precursors: PbI2 and FAI
* Alternative solvents: IPA, n-BuOH, t-BuOH
* Molar ratios: 3:4 to 5:4 (Nitu2022 pages 4-6)
* Concentrations: Not specified
* Purity requirements: Not mentioned
* Primary solvents: DMF (Cimrov2023 pages 4-4, Han2016 pages 5-5, Cimrov2023 pages 3-4)
* Solvent ratios for mixed systems: Not specified

**Dissolution conditions:**

* Temperature: 60°C (Han2016 pages 5-5), 70°C (Cimrov2023 pages 3-4)
* Time: Overnight (Han2016 pages 5-5), 30 minutes (Cimrov2023 pages 3-4)

**Processing parameters:**

* Spin-coating speeds and times: 1500-2500 rpm, 15 s (Cimrov2023 pages 4-4), 3000 rpm (Cimrov2023 pages 3-4)
* Spin-coating steps: Two-step sequential method (Cimrov2023 pages 3-4, Cimrov2023 pages 4-4), One-step (Han2016 pages 5-5)
* Substrate temperatures: Not specified
* Annealing temperatures and durations: 170°C f

In [12]:
# Query for related compound synthesis infomation
related_compounds_query = """
Extract synthesis insights for related APbI3 compounds:

PRECURSORS:
- Common solute chemicals across different compositions
- Molar ratios and concentrations that work for different compositions
- Solvent preferences for different precursors (DMF, DMSO, others)
- Solvent ratios for mixed systems
- General dissolution conditions (temperature, time)

PROCESSING PARAMETERS RANGES TYPICAL FOR APbI3 FAMILY:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

PERFORMANCE CORRELATIONS:
- How synthesis conditions affect film quality including morphology, uniformity, thickness
- Common challenges and solutions

TRANSFERABLE KNOWLEDGE:
- Which parameters can be adapted from related compounds
- Best practices that apply broadly

Focus on synthesis wisdom that could guide FAPbI3 optimization.
"""

# Execute analysis
related_answer = run_local_analysis(related_compounds_query, pdf_dir, "Related APbI3 Synthesis")

# Display results
if related_answer:
    print("\n" + "="*60)
    print("RELATED COMPOUND SYNTHESIS INSIGHTS")
    print("="*60)
    
    # Extract answer
    if hasattr(related_answer, 'answer'):
        answer_text = related_answer.answer
    elif hasattr(related_answer, 'formatted_answer'):
        answer_text = related_answer.formatted_answer
    else:
        answer_text = str(related_answer)
    
    print(answer_text)
    
    # Show evidence summary
    if hasattr(related_answer, 'contexts') and related_answer.contexts:
        print(f"\n Evidence: {len(related_answer.contexts)} relevant passages found")
    else:
        print("\n No comparative evidence found")
    
    print("\n✅ Related compounds analysis complete!")
else:
    print("❌ Related compounds analysis failed")

print("\n" + "="*40)
print("Ready for optimization strategies analysis...")

Starting: Related APbI3 Synthesis
Analyzing papers in: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Processing 23 research papers...
  1/23 Loading: pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00...


16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:37 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  2/23 Loading: pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10...


16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:20:48 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  3/23 Loading: document.pdf...


...

  23/23 Loading: Small Methods - 2018 - Li - Formamidinium%E2%80%90...


16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:25:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

Analyzing with local LLM...


16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:26:17 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

Related APbI3 Synthesis completed!

RELATED COMPOUND SYNTHESIS INSIGHTS
Synthesis Insights for Related APbI3 Compounds:

PRECURSORS:
The use of DMF as a common solvent across different compositions is evident in the studies. Additionally, the molar ratio of FAI:PbI2 at 1:1 is mentioned in both papers (Yang2020 pages 3-3, Sampson2019 pages 2-2). Solvent preferences for different precursors are also discussed, with DMF and DMSO being used in conjunction with other solvents like IPA (Sampson2019 pages 2-2).

Molar ratios and concentrations that work for different compositions are not explicitly stated, but the use of a 20 mg/mL concentration solution of FAI is mentioned (Sampson2019 pages 2-2). Solvent ratios for mixed systems are also discussed, with temperatures ranging from 10% to 100% NMP being used in one study (Yang2020 pages 3-3).

PROCESSING PARAMETERS RANGES TYPICAL FOR APbI3 FAMILY:
Spin-coating speeds and times vary between studies, but a spin program at 1000 rpm for 10 s and 4

In [13]:
# Query for detailed element-based analysis
element_base_analysis_query = """
Extract detailed synthesis information about precursors and processing parameters for compounds containing Formamidinium(FA), Lead(Pb), or Iodine(I):

PRECURSOR:
- Common source chemicals for FA, Pb, I
- Common solvents or mixed solvent systems and ratios used for these solutes
- Precursor dissolving conditions and concentrations (stirring and heating requirements, concentrations optimization ranges)
- Precursor stability and storage conditions
- Purity requirements and impurity effects
- Order of addition effects

PROCESSING PARAMETERS RANGES TYPICAL FOR COMPOUNDS CONTAINING FA, Pb, I RESPECTIVELY:
- Spin-coating speeds and times
- Spin-coating steps (one-step vs multi-step, with or without antisolvent)
- Substrate temperatures (with or without preheating)
- Annealing temperatures and durations
- Atmosphere requirements (N2, air, vacuum)

TRANSFERABLE KNOWLEDGE:
- Which parameters can be adapted from compounds containing FA, Pb, I
- Best practices that apply broadly

Provide specific numerical data and practical guidelines.
"""

# Execute analysis
element_base_answer = run_local_analysis(element_base_analysis_query, pdf_dir, "Element-Based Analysis")

# Display results
if element_base_answer:
    print("\n" + "="*60)
    print("ELEMENT-BASED ANALYSIS")
    print("="*60)
    
    # Extract answer
    if hasattr(element_base_answer, 'answer'):
        answer_text = element_base_answer.answer
    elif hasattr(element_base_answer, 'formatted_answer'):
        answer_text = element_base_answer.formatted_answer
    else:
        answer_text = str(element_base_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(element_base_answer, 'contexts') and element_base_answer.contexts:
        print(f"\n Evidence: {len(element_base_answer.contexts)} relevant passages found")
    else:
        print("\n No element-based evidence found")

    print("\n✅ Element-based analysis complete!")
else:
    print("❌ Element-based analysis failed")

Starting: Element-Based Analysis
Analyzing papers in: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Processing 23 research papers...
  1/23 Loading: pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00...


16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:27:59 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  2/23 Loading: pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10...


16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:28:12 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  3/23 Loading: document.pdf...


...

  23/23 Loading: Small Methods - 2018 - Li - Formamidinium%E2%80%90...


16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:35 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

Analyzing with local LLM...


16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:32:57 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

Element-Based Analysis completed!

ELEMENT-BASED ANALYSIS
**Precursors**

Formamidinium (FA), Lead (Pb), and Iodine (I) precursors are commonly sourced from various chemicals. Formamidinium acetate (FAac) is a common source for FA, while lead nitrate (Pb(NO3)2) and potassium iodide (KI) are used as sources for Pb and I, respectively. Solvents such as tert-butanol (t-BuOH), IPA, n-BuOH, and gamma-butyrolactone (GBL) are used in varying ratios to dissolve these precursors.

Precursor dissolving conditions and concentrations are crucial for successful synthesis. For FAI, a 50°C bath with continuous stirring is recommended (Firoz2020 pages 5-5). The concentration of PbI2 in DMF can be optimized between 0.1-1.0 M (Cimrov2023 pages 8-9). The order of addition effects the final product; for example, adding FAI to PbI2 requires continuous stirring (Cimrov2023 pages 7-8).

**Processing Parameters**

Processing parameters for compounds containing FA, Pb, and I are as follows:

* Spin-coating spe

In [14]:
# Query for other synthesis notes
other_notes_query = """
Extract other synthesis details and best practices for FAPbI3 and related compounds:

PROCESSING DETAILS:
- Pre-synthesis treatments
- Post-synthesis treatments
- Additive strategies

CHARACTERIZATION METHODS:
- Essential techniques for evaluating synthesis quality
- Key metrics and target values

TROUBLESHOOTING GUIDE:
- Common synthesis problems and solutions
- Film quality issues (defects, nonuniformity and impurity phases) and their causes
- Process robustness improvements

Focus on small details which are important for synthesis quality and proven best practices.
"""

# Execute analysis
other_notes_answer = run_local_analysis(other_notes_query, pdf_dir, "Other Synthesis Notes")

# Display results
if other_notes_answer:
    print("\n" + "="*60)
    print("OTHER SYNTHESIS NOTES")
    print("="*60)
    
    # Extract answer
    if hasattr(other_notes_answer, 'answer'):
        answer_text = other_notes_answer.answer
    elif hasattr(other_notes_answer, 'formatted_answer'):
        answer_text = other_notes_answer.formatted_answer
    else:
        answer_text = str(other_notes_answer)

    print(answer_text)
    
    # Show evidence summary
    if hasattr(other_notes_answer, 'contexts') and other_notes_answer.contexts:
        print(f"\n Evidence: {len(other_notes_answer.contexts)} relevant passages found")
    else:
        print("\n No other synthesis evidence found")

    print("\n✅ Other synthesis notes analysis complete!")
else:
    print("❌ Other synthesis notes analysis failed")


Starting: Other Synthesis Notes
Analyzing papers in: /Users/shengfang/Desktop/TRI/test_FAPbI3/pdfs
Processing 23 research papers...
  1/23 Loading: pubs.rsc.org_en_content_articlehtml_2015_mh_c5mh00...


16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  2/23 Loading: pubs.rsc.org_en_content_articlehtml_2019_nr_c8nr10...


16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:34:39 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

  3/23 Loading: document.pdf...


...

  23/23 Loading: Small Methods - 2018 - Li - Formamidinium%E2%80%90...


16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:29 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. 

Analyzing with local LLM...


16:39:50 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:50 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:50 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:50 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
16:39:50 - LiteLLM:WARNING: logging_callback_manager.py:130 - Cannot add callback - would exceed MAX_CALLBACKS limit of 30. Current callbacks: 30
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollama/llama3.2.
Could not find cost for model ollam

Other Synthesis Notes completed!

OTHER SYNTHESIS NOTES
**Processing Details**

Pre-synthesis treatments include cleaning glass substrates to ensure surface quality and prevent contamination (Firoz2020 pages 5-5). Post-synthesis treatments involve drying and recrystallization to remove excess solvent and impurities (Firoz2020 pages 5-5).

Additive strategies are crucial for improving synthesis quality. The use of thiourea as an additive has been shown to control crystallization and morphology, leading to high-quality FAPbI3 films (Yang2020 pages 3-3). Lewis base properties and hydrogen-bond-accepting ability are also essential in achieving pure α-phase films (Yang2020 pages 3-3).

**Characterization Methods**

Essential techniques for evaluating synthesis quality include cross-sectional SEM images, PCE measurements under continuous light illumination, absorption and PL spectra analysis (Li2018 pages 16-17). Key metrics and target values include crystal structure, purity, and film unifo

In [15]:
# Get target composition from previous analysis (with fallback)
try:
    target_comp = TARGET_COMPOSITION
except NameError:
    target_comp = "Target_Composition"  # Fallback if variable not defined

print(f" Target Composition: {target_comp}")

# Collect all analysis results
all_answers = []
section_titles = [
    f"Direct {target_comp} Synthesis Parameters",
    "Related Compound Synthesis Insights", 
    "Element-Based Synthesis Analysis",
    "Other Synthesis Notes"
]

# Add results from each analysis (with fallback for missing variables)
results = []
try:
    if 'focused_result' in locals() and focused_result:
        results.append(('focused_result', focused_result))
except: pass

try:
    if 'related_answer' in locals() and related_answer:
        results.append(('related_answer', related_answer))
except: pass

try:
    if 'element_base_answer' in locals() and element_base_answer:
        results.append(('element_base_answer', element_base_answer))
except: pass

try:
    if 'other_notes_answer' in locals() and other_notes_answer:
        results.append(('other_notes_answer', other_notes_answer))
except: pass

# Create comprehensive summary
results_summary = {
    "target_composition": target_comp,
    "source_documents": len(glob.glob(os.path.join(pdf_dir, "*.pdf"))) if 'pdf_dir' in locals() else 0,
    "pdf_directory": pdf_dir if 'pdf_dir' in locals() else "Not specified",
    "total_sections": len(section_titles),
    "sections": {}
}

print(f"Processing {len(results)} analysis sections...")

# Process each result with Unicode-safe handling
for i, (var_name, answer) in enumerate(results):
    title = section_titles[i] if i < len(section_titles) else f"Analysis {i+1}"
    
    try:
        # Extract answer text safely
        if hasattr(answer, 'answer'):
            raw_text = str(answer.answer)
        elif hasattr(answer, 'formatted_answer'):
            raw_text = str(answer.formatted_answer)
        else:
            raw_text = str(answer)
        
        # Clean problematic Unicode characters
        clean_text = raw_text.encode('utf-8', errors='replace').decode('utf-8', errors='replace')
        
        # Store in summary
        results_summary["sections"][title] = {
            "content": clean_text,
            "evidence_count": len(answer.contexts) if hasattr(answer, 'contexts') and answer.contexts else 0,
            "analysis_variable": var_name
        }
        
        print(f"{title}: {len(clean_text)} characters")
        
    except Exception as e:
        print(f"Error processing {title}: {str(e)}")
        results_summary["sections"][title] = {
            "content": f"Error processing this section: {str(e)}",
            "evidence_count": 0,
            "analysis_variable": var_name
        }

# Save comprehensive results with composition-specific filename
safe_comp_name = target_comp.replace('3', '3').replace('2', '2').lower()  # Clean filename
results_file = f"/Users/shengfang/Desktop/TRI/{safe_comp_name}_synthesis_knowledge.json"

try:
    import json
    with open(results_file, 'w', encoding='utf-8') as f:
        json.dump(results_summary, f, indent=2, ensure_ascii=False)
    
    print(f"\n Comprehensive results saved to: {results_file}")
    
    # Display summary
    print("\n" + "="*70)
    print(f"COMPREHENSIVE {target_comp} SYNTHESIS KNOWLEDGE SUMMARY")
    print("="*70)
    
    print(f" Analysis Statistics:")
    print(f"   • Target Composition: {target_comp}")
    print(f"   • Source Documents: {results_summary['source_documents']} PDFs")
    print(f"   • Analysis Sections: {results_summary['total_sections']}")
    print(f"   • Processed Sections: {len(results_summary['sections'])}")
    
    # Show brief section overview
    for title, data in results_summary["sections"].items():
        content_preview = data["content"][:150] if data["content"] else "No content"
        print(f"\n {title}:")
        print(f"    Evidence Sources: {data['evidence_count']}")
        print(f"    Content Preview: {content_preview}...")
    
    print(f"\n✅ Complete synthesis knowledge successfully compiled!")
    print(f" Full results available in: {results_file}")
    
except Exception as e:
    print(f"❌ Error saving results: {str(e)}")

print("\n" + "="*50)
print(f" {target_comp} Analysis Complete!")
print("All synthesis knowledge has been extracted and organized.")

 Target Composition: FAPbI3
Processing 4 analysis sections...
Direct FAPbI3 Synthesis Parameters: 1344 characters
Related Compound Synthesis Insights: 1800 characters
Element-Based Synthesis Analysis: 1690 characters
Other Synthesis Notes: 1697 characters

 Comprehensive results saved to: /Users/shengfang/Desktop/TRI/fapbi3_synthesis_knowledge.json

COMPREHENSIVE FAPbI3 SYNTHESIS KNOWLEDGE SUMMARY
 Analysis Statistics:
   • Target Composition: FAPbI3
   • Source Documents: 23 PDFs
   • Analysis Sections: 4
   • Processed Sections: 4

 Direct FAPbI3 Synthesis Parameters:
    Evidence Sources: 10
    Content Preview: The synthesis details for FAPbI3 thin films are as follows:

**Precurors:**

* Primary precursors: PbI2 and FAI
* Alternative solvents: IPA, n-BuOH, t...

 Related Compound Synthesis Insights:
    Evidence Sources: 10
    Content Preview: Synthesis Insights for Related APbI3 Compounds:

PRECURSORS:
The use of DMF as a common solvent across different compositions is evident i

In [2]:
import ollama
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from emukit.core import ParameterSpace, ContinuousParameter, DiscreteParameter
from emukit.core.initial_designs.random_design import RandomDesign
from emukit.core.initial_designs.latin_design import LatinDesign

In [3]:
def x_normalizer(X, var_array):
    
    def max_min_scaler(x, x_max, x_min):
        return (x-x_min)/(x_max-x_min)
    x_norm = []
    for x in (X):
           x_norm.append([max_min_scaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_norm

def x_denormalizer(x_norm, var_array):
    
    def max_min_rescaler(x, x_max, x_min):
        return x*(x_max-x_min)+x_min
    x_original = []
    for x in (x_norm):
           x_original.append([max_min_rescaler(x[i], 
                                         max(var_array[i]), 
                                         min(var_array[i])) for i in range(len(x))])
            
    return x_original

def get_closest_value(given_value, array_list):
    absolute_difference_function = lambda list_value : abs(list_value - given_value)
    closest_value = min(array_list, key=absolute_difference_function)
    return closest_value
    
def get_closest_array(suggested_x, var_list):
    modified_array = []
    for x in suggested_x:
        modified_array.append([get_closest_value(x[i], var_list[i]) for i in range(len(x))])
    return np.array(modified_array)

In [18]:
def load_synthesis_knowledge(json_path):
    """Load synthesis knowledge from JSON file"""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            knowledge = json.load(f)
        print(f"✅ Loaded synthesis knowledge for {knowledge.get('target_composition', 'Unknown')}")
        return knowledge
    except FileNotFoundError:
        print(f"❌ File not found: {json_path}")
        return None
    except json.JSONDecodeError:
        print(f"❌ Invalid JSON format in: {json_path}")
        return None

def extract_parameter_insights(knowledge, method="filtered"):
    """Extract synthesis insights from knowledge base
    
    Args:
        knowledge: Loaded JSON knowledge base
        method: "filtered" (clean format) or "raw" (full JSON as string)
    """
    
    if method == "raw":
        # Option 1: Send entire JSON as string
        return json.dumps(knowledge, indent=2)
    
    elif method == "filtered":
        # Option 2: Extract and format only synthesis content (current approach)
        insights = []
        
        if 'sections' in knowledge:
            for section_name, section_data in knowledge['sections'].items():
                content = section_data.get('content', '')
                if content and content != 'No content':
                    insights.append(f"## {section_name}\n{content}\n")
        
        return "\n".join(insights)
    
    else:
        raise ValueError("Method must be 'filtered' or 'raw'")

def generate_parameter_space_llm(synthesis_insights, target_params):
    """Use local LLM to propose parameter optimization space"""
    
    prompt = f"""
Based on the following synthesis knowledge for FAPbBr3, propose SPECIFIC NUMERICAL RANGES AND STEP SIZES for optimization parameters.

SYNTHESIS KNOWLEDGE:
{synthesis_insights}

TARGET PARAMETERS TO OPTIMIZE:
{target_params}

Please provide SPECIFIC NUMERICAL RANGES AND STEP SIZES for each parameter based on the literature evidence:


   
1. **Spin Speed**: 
   - Range: in rpm (e.g., 500 - 5000)
   - Step Size: Increment between values (e.g., 500 rpm)
   
2. **Precursor Concentration**: 
   - Range: in mol/L (e.g., 0.1 - 1.5)
   - Step Size: Increment between values (e.g., 0.1 mol/L)
   
3. **Annealing Temperature**: 
   - Range: in °C (e.g., 100 - 300)
   - Step Size: Increment between values (e.g., 10°C)

For each parameter, provide:
- Recommended minimum value
- Recommended maximum value
- Recommended step size (increment between discrete values)
- Rationale based on the synthesis knowledge
- Key literature insights that support the range and resolution

Format your response clearly with specific numbers that can be used for Bayesian optimization setup.
"""

    try:
        print("Querying local Llama3.2 for parameter space recommendations...")
        response = ollama.chat(
            model='llama3.2',
            messages=[{
                'role': 'user', 
                'content': prompt
            }],
            options={
                'temperature': 0.1, 
                'top_p': 0.9,
                'num_predict': 2048
            }
        )
        
        return response['message']['content']
        
    except Exception as e:
        print(f"❌ Error querying LLM: {e}")
        return None

def parse_llm_recommendations(llm_output):
    """Parse LLM output to extract numerical parameter ranges"""
    
    print("\n" + "="*60)
    print("LLM PARAMETER SPACE RECOMMENDATIONS")
    print("="*60)
    print(llm_output)
    print("="*60)  
    return llm_output


# Load the synthesis knowledge JSON file
knowledge_path = "/Users/shengfang/Desktop/TRI/fapbi3_synthesis_knowledge.json"
synthesis_knowledge = load_synthesis_knowledge(knowledge_path)

if synthesis_knowledge:
    # Choose extraction method
    extraction_method = "filtered"  # Change to "raw" to send full JSON
    
    # Extract insights using chosen method
    insights = extract_parameter_insights(synthesis_knowledge, method=extraction_method)
    
    # Define target parameters from your optimization space
    target_parameters = """
    1. Spin Speed
    2. Precursor Concentration
    3. Annealing Temperature
    """
    
    print(f"Method: {extraction_method}")
    print(f"Analyzing {len(insights)} characters of synthesis knowledge...")

    # Generate LLM recommendations
    llm_recommendations = generate_parameter_space_llm(insights, target_parameters)
    if llm_recommendations:
        parsed_recommendations = parse_llm_recommendations(llm_recommendations)
    else:
        print("❌ No recommendations received from LLM")
        
else:
    print("❌ Could not load synthesis knowledge - using default parameter ranges")

✅ Loaded synthesis knowledge for FAPbI3
Method: filtered
Analyzing 6676 characters of synthesis knowledge...
Querying local Llama3.2 for parameter space recommendations...

LLM PARAMETER SPACE RECOMMENDATIONS
Based on the provided synthesis knowledge, here are the specific numerical ranges and step sizes for each parameter:

1. **Spin Speed**:
   - Range: 500-2500 rpm
   - Step Size: Increment of 500 rpm (e.g., 500, 1000, 1500, 2000, 2500)
   - Recommended Minimum Value: 500 rpm (Cimrov2023 pages 4-4)
   - Recommended Maximum Value: 2500 rpm (Cimrov2023 pages 4-4)
   - Rationale: The literature suggests that spin-coating speeds between 500-2500 rpm can produce high-quality FAPbI3 films. A step size of 500 rpm allows for a balanced exploration of this range.
   - Key Literature Insight: Cimrov2023 pages 4-4 mentions an optimized condition at 1000 rpm, but the full range is explored to identify optimal conditions.

2. **Precursor Concentration**:
   - Range: 0.1-1.5 mol/L
   - Step Size:

In [5]:
# After reviewing LLM recommendations, manually define the parameter space
spinspeed_min, spinspeed_max, spinspeed_step = [500, 3000, 500] ## Unit: rpm
spinspeed_var = np.arange(spinspeed_min, spinspeed_max+spinspeed_step, spinspeed_step) 
spinspeed_num = len(spinspeed_var)

concentration_min, concentration_max, concentration_step = [0.1, 1.6, 0.1] ## Unit: mol/L
concentration_var = np.arange(concentration_min, concentration_max+concentration_step, concentration_step)
concentration_num = len(concentration_var)

annealingtemp_min, annealingtemp_max, annealingtemp_step = [100, 310, 10] # Unit: degC
annealingtemp_var = np.arange(annealingtemp_min, annealingtemp_max+annealingtemp_step, annealingtemp_step)
annealingtemp_num = len(annealingtemp_var)


var_array = [spinspeed_var, concentration_var, 
             annealingtemp_var]
x_labels = ['Spin Speed [rpm]',  
            'Precursor Concentration [mol/L]', 
            'Annealing Temperature [degC]']

random_state = np.random.RandomState(42)
parameter_space = ParameterSpace([ContinuousParameter('x1', 0, 1),
                                 ContinuousParameter('x2', 0, 1),
                                 ContinuousParameter('x3', 0, 1),
                                 ])

# parameter_space = ParameterSpace([DiscreteParameter('x1', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x2', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x3', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x4', np.linspace(0,1, 51)),
#                                  DiscreteParameter('x5', np.linspace(0,1, 51)),
#                                  ])
    

# Generate initial samples with latin hypercube
design = LatinDesign(parameter_space)
x_init = design.get_samples(10)
x_init_original = get_closest_array(x_denormalizer(x_init, var_array),var_array)

df = pd.DataFrame(x_init_original, columns = x_labels)
df_cols = x_labels
df.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/initial_samples_10_latin_plus.csv", index=True)

In [8]:
# Save to lists for easier LLM processing 
sample_conditions = []
for i, (index, row) in enumerate(df.iterrows()):
    condition = {
        'sample_id': f"Sample_{i+1}",
        'spin_speed': row[x_labels[0]],  # Spin Speed [rpm]
        'concentration': row[x_labels[1]],  # Precursor Concentration [mol/L]
        'annealing_temp': row[x_labels[2]]  # Annealing Temperature [degC]
    }
    sample_conditions.append(condition)

def llm_filter_samples(synthesis_knowledge, sample_conditions, target_count=6):
    """Use LLM to filter samples based on synthesis knowledge confidence"""
    
    # Format samples for LLM
    samples_text = ""
    for i, condition in enumerate(sample_conditions, 1):
        samples_text += f"""
Sample {i}:
- Spin Speed: {condition['spin_speed']:.0f} rpm
- Concentration: {condition['concentration']:.3f} mol/L
- Annealing Temperature: {condition['annealing_temp']:.0f}°C
"""

    # Get synthesis insights
    insights = extract_parameter_insights(synthesis_knowledge, method="filtered")
    
    prompt = f"""
Based on the following FAPbI3 synthesis knowledge, evaluate and rank the experimental conditions below. 

SYNTHESIS KNOWLEDGE:
{insights}

EXPERIMENTAL CONDITIONS TO EVALUATE:
{samples_text}

Please select the TOP {target_count} experimental conditions that are most likely to produce high-quality alpha-phase FAPbI3 films based on the literature evidence.

For each selected condition, provide:
1. **Sample ID** (e.g., Sample 1, Sample 2, etc.)
2. **Confidence Score** (0-1, where 1 = highest confidence)
3. **Rationale** explaining why this condition is promising based on the synthesis knowledge

Consider:
- Parameter combinations that align with successful literature reports
- Avoiding extreme values that might cause processing issues
- Synergistic effects between parameters (e.g., solvent ratio + concentration)
- Temperature compatibility with precursor stability
- Spin speed effects on film uniformity

Rank them from highest to lowest confidence and provide ONLY the top {target_count} conditions.

Format your response clearly listing each selected sample with its confidence score and rationale.
"""

    try:
        print(f"Querying LLM to select top {target_count} experimental conditions...")
        response = ollama.chat(
            model='llama3.2',
            messages=[{
                'role': 'user',
                'content': prompt
            }],
            options={
                'temperature': 0.1,  
                'top_p': 1.0,
                'num_predict': 2048
            }
        )
        
        return response['message']['content']
        
    except Exception as e:
        print(f"❌ Error querying LLM for sample filtering: {e}")
        return None

def load_synthesis_knowledge(json_path):
    """Load synthesis knowledge from JSON file"""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            knowledge = json.load(f)
        print(f"✅ Loaded synthesis knowledge for {knowledge.get('target_composition', 'Unknown')}")
        return knowledge
    except FileNotFoundError:
        print(f"❌ File not found: {json_path}")
        return None
    except json.JSONDecodeError:
        print(f"❌ Invalid JSON format in: {json_path}")
        return None

def extract_parameter_insights(knowledge, method="filtered"):
    """Extract synthesis insights from knowledge base
    
    Args:
        knowledge: Loaded JSON knowledge base
        method: "filtered" (clean format) or "raw" (full JSON as string)
    """
    
    if method == "raw":
        # Option 1: Send entire JSON as string
        return json.dumps(knowledge, indent=2)
    
    elif method == "filtered":
        # Option 2: Extract and format only synthesis content (current approach)
        insights = []
        
        if 'sections' in knowledge:
            for section_name, section_data in knowledge['sections'].items():
                content = section_data.get('content', '')
                if content and content != 'No content':
                    insights.append(f"## {section_name}\n{content}\n")
        
        return "\n".join(insights)
    
    else:
        raise ValueError("Method must be 'filtered' or 'raw'")

# Load the synthesis knowledge JSON file
knowledge_path = "/Users/shengfang/Desktop/TRI/test_FAPbI3/fapbi3_synthesis_knowledge.json"
synthesis_knowledge = load_synthesis_knowledge(knowledge_path)


if synthesis_knowledge:
    llm_filtering_result = llm_filter_samples(synthesis_knowledge, sample_conditions, target_count=6)
    
    if llm_filtering_result:
        print("LLM SAMPLE SELECTION RESULTS:")
        print("="*100)
        print(llm_filtering_result)
        print("="*100)
        
        # Parse LLM response to extract selected sample IDs (only numbered selections, not mentions)
        import re
        # Look for patterns like "1. **Sample X**" or "1. Sample X" (actual selections)
        sample_pattern = r'^\d+\.\s+\*\*Sample\s+(\d+)\*\*|^\d+\.\s+Sample\s+(\d+)'
        selected_sample_ids = []
        
        for line in llm_filtering_result.split('\n'):
            match = re.search(sample_pattern, line.strip())
            if match:
                # Get the captured group that's not None
                sample_num = int(match.group(1) if match.group(1) else match.group(2))
                if sample_num <= len(df):  # Ensure sample ID is valid
                    selected_sample_ids.append(sample_num - 1)  # Convert to 0-based index
        
        # Remove duplicates and sort
        selected_sample_ids = sorted(list(set(selected_sample_ids)))
        
        
        if selected_sample_ids:
            # Create filtered DataFrame with LLM-selected samples
            filter_df = df.iloc[selected_sample_ids].copy()
            filter_df.reset_index(drop=True, inplace=True)  # Reset index for clean display
            
            print(f"\n FILTERED SAMPLES (LLM SELECTED - {len(filter_df)} samples):")
            print("="*100)
            
            
            # Format for clean display
            print(filter_df.round(3).to_string(index=False, float_format='%.3f'))
            print("="*100)
            
            # Save filtered samples
            filter_df.to_csv("/Users/shengfang/Desktop/TRI/test_FAPbI3/filtered_samples_llm_selected_plus.csv", index=True)
            
            
        else:
            print("\n No valid sample IDs extracted from LLM response - using first 6 samples as fallback")
            filter_df = df.head(6).copy()
            filter_df.reset_index(drop=True, inplace=True)
    
        
    else:
        print("❌ Failed to get LLM filtering results")
else:
    print("❌ No synthesis knowledge available for filtering")
    print("="*100)

✅ Loaded synthesis knowledge for FAPbI3
Querying LLM to select top 6 experimental conditions...
LLM SAMPLE SELECTION RESULTS:
Based on the synthesis knowledge provided, I have evaluated the experimental conditions and ranked the top 6 conditions that are most likely to produce high-quality alpha-phase FAPbI3 films. Here are the results:

1. **Sample 4**
   - Spin Speed: 2500 rpm
   - Concentration: 1.500 mol/L
   - Annealing Temperature: 150°C
   - Confidence Score: 1
   - Rationale: This condition aligns with successful literature reports, as it uses a moderate spin speed and high concentration of PbI2 in DMF, which is known to promote α-phase FAPbI3 formation. The annealing temperature is also within the recommended range for precursor stability.

2. **Sample 1**
   - Spin Speed: 1500 rpm
   - Concentration: 1.100 mol/L
   - Annealing Temperature: 110°C
   - Confidence Score: 0.9
   - Rationale: This condition uses a moderate spin speed and high concentration of PbI2 in DMF, which is

In [1]:
# =============================================================================
# COMPREHENSIVE MODEL PERFORMANCE ANALYSIS
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import cross_val_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
import warnings
warnings.filterwarnings('ignore')

def comprehensive_model_analysis():
    """
    Comprehensive model performance analysis including:
    1. Cross-validation performance
    2. Prediction accuracy
    3. Uncertainty calibration
    4. Parameter space coverage
    5. Constraint satisfaction analysis
    6. Multi-objective trade-offs
    """
    
    print("🔬 COMPREHENSIVE MODEL PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # ====================
    # 1. DATA PREPARATION
    # ====================
    
    # Extract features and targets
    X_features = X_train[param_names].values
    y_coverage = y_train_raw['coverage'].values
    y_uniformity = y_train_raw['uniformity'].values
    y_phase_purity = y_train_raw['phase_purity'].values
    
    print(f"📊 Dataset Overview:")
    print(f"   • Training samples: {len(X_train)}")
    print(f"   • Parameters: {len(param_names)}")
    print(f"   • Objectives: coverage (max), uniformity (min), phase_purity (constraint)")
    
    # ====================
    # 2. CROSS-VALIDATION ANALYSIS
    # ====================
    
    print(f"\n📈 Cross-Validation Performance:")
    
    # Test different GP kernels
    kernels = {
        'RBF': RBF(),
        'Matern_5/2': Matern(nu=2.5),
        'Matern_3/2': Matern(nu=1.5),
        'RBF + Noise': RBF() + WhiteKernel()
    }
    
    cv_results = {}
    
    for obj_name, y_target in [('coverage', y_coverage), ('uniformity', y_uniformity)]:
        print(f"\n   🎯 {obj_name.upper()} Prediction:")
        cv_results[obj_name] = {}
        
        for kernel_name, kernel in kernels.items():
            try:
                gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=3)
                scores = cross_val_score(gp, X_features, y_target, cv=min(5, len(X_train)//2), scoring='r2')
                cv_results[obj_name][kernel_name] = {
                    'r2_mean': scores.mean(),
                    'r2_std': scores.std(),
                    'scores': scores
                }
                
                print(f"      {kernel_name:15s}: R² = {scores.mean():.3f} ± {scores.std():.3f}")
                
            except Exception as e:
                print(f"      {kernel_name:15s}: Failed ({str(e)[:30]}...)")
                cv_results[obj_name][kernel_name] = {'r2_mean': 0, 'r2_std': 0, 'scores': []}
    
    # ====================
    # 3. PREDICTION ACCURACY ON TRAINING DATA
    # ====================
    
    print(f"\n🎯 Training Data Fit Quality:")
    
    try:
        # Get model predictions for training data if possible
        if hasattr(ax_client.generation_strategy, '_model') and ax_client.generation_strategy._model is not None:
            model_bridge = ax_client.generation_strategy._model
            
            # Create observation features for training data
            from ax.core.observation import ObservationFeatures
            train_obs_features = []
            for i in range(len(X_train)):
                norm_params = {f"{param}_norm": X_train_norm.iloc[i][f"{param}_norm"] for param in param_names}
                train_obs_features.append(ObservationFeatures(parameters=norm_params))
            
            # Get predictions
            train_predictions = model_bridge.predict(train_obs_features)
            train_means, train_covariances = train_predictions
            
            # Analyze prediction quality
            for obj_name in train_means.keys():
                predicted = [float(train_means[obj_name][i]) for i in range(len(train_means[obj_name]))]
                
                if obj_name == 'coverage':
                    actual = y_coverage
                elif obj_name == 'uniformity':
                    actual = y_uniformity
                else:
                    continue
                
                # Calculate metrics
                r2 = r2_score(actual, predicted)
                mse = mean_squared_error(actual, predicted)
                mae = mean_absolute_error(actual, predicted)
                correlation, p_val = pearsonr(actual, predicted)
                
                print(f"   📊 {obj_name.upper()}:")
                print(f"      R² Score: {r2:.4f}")
                print(f"      RMSE: {np.sqrt(mse):.4f}")
                print(f"      MAE: {mae:.4f}")
                print(f"      Correlation: {correlation:.4f} (p={p_val:.4f})")
                
                # Calculate prediction intervals
                uncertainties = []
                for i in range(len(train_covariances[obj_name])):
                    cov = train_covariances[obj_name][i]
                    if hasattr(cov, 'item'):
                        std = abs(float(cov.item())) ** 0.5
                    else:
                        std = abs(float(cov)) ** 0.5 if isinstance(cov, (int, float)) else 0.0
                    uncertainties.append(std)
                
                avg_uncertainty = np.mean(uncertainties)
                print(f"      Avg Uncertainty: {avg_uncertainty:.4f}")
                
        else:
            print("   ⚠️ Model bridge not available for training predictions")
            
    except Exception as e:
        print(f"   ❌ Training prediction analysis failed: {e}")
    
    # ====================
    # 4. PARAMETER SPACE ANALYSIS
    # ====================
    
    print(f"\n🗺️ Parameter Space Coverage:")
    
    # Analyze parameter distributions
    for i, param in enumerate(param_names):
        values = X_train[param].values
        param_range = param_ranges[param]
        
        coverage = (values.max() - values.min()) / (param_range[1] - param_range[0])
        density = len(np.unique(values)) / len(values)
        
        print(f"   📏 {param}:")
        print(f"      Range Coverage: {coverage:.1%} of full range")
        print(f"      Value Density: {density:.1%} (uniqueness)")
        print(f"      Distribution: {values.min():.1f} to {values.max():.1f}")
    
    # ====================
    # 5. CONSTRAINT SATISFACTION ANALYSIS
    # ====================
    
    print(f"\n🔒 Constraint Satisfaction Analysis:")
    
    phase_purity_success = (y_train_raw['phase_purity'] == 1).sum()
    phase_purity_rate = phase_purity_success / len(y_train_raw)
    
    print(f"   • Phase Purity Constraint Success: {phase_purity_success}/{len(y_train_raw)} ({phase_purity_rate:.1%})")
    print(f"   • Feasible Experiments: {phase_purity_success} out of {len(y_train_raw)}")
    
    # Analyze constraint violation distribution
    violations = y_train_raw['phase_purity_violation'].values
    feasible_mask = violations <= 0
    
    print(f"   • Constraint Violations: min={violations.min():.2f}, max={violations.max():.2f}")
    print(f"   • Feasible Region: {feasible_mask.sum()}/{len(feasible_mask)} experiments")
    
    # ====================
    # 6. MULTI-OBJECTIVE TRADE-OFF ANALYSIS
    # ====================
    
    print(f"\n⚖️ Multi-Objective Trade-off Analysis:")
    
    # Pareto frontier analysis
    feasible_coverage = y_train_raw[feasible_mask]['coverage'].values
    feasible_uniformity = y_train_raw[feasible_mask]['uniformity'].values
    
    if len(feasible_coverage) > 1:
        # Find Pareto optimal points
        pareto_mask = []
        for i in range(len(feasible_coverage)):
            is_pareto = True
            for j in range(len(feasible_coverage)):
                if i != j:
                    # Check if point j dominates point i
                    if (feasible_coverage[j] >= feasible_coverage[i] and 
                        feasible_uniformity[j] <= feasible_uniformity[i] and
                        (feasible_coverage[j] > feasible_coverage[i] or feasible_uniformity[j] < feasible_uniformity[i])):
                        is_pareto = False
                        break
            pareto_mask.append(is_pareto)
        
        pareto_points = np.sum(pareto_mask)
        pareto_coverage = feasible_coverage[pareto_mask]
        pareto_uniformity = feasible_uniformity[pareto_mask]
        
        print(f"   • Pareto Optimal Points: {pareto_points}/{len(feasible_coverage)} feasible experiments")
        print(f"   • Best Coverage: {feasible_coverage.max():.3f}")
        print(f"   • Best Uniformity: {feasible_uniformity.min():.3f}")
        
        if pareto_points > 0:
            print(f"   • Pareto Coverage Range: {pareto_coverage.min():.3f} - {pareto_coverage.max():.3f}")
            print(f"   • Pareto Uniformity Range: {pareto_uniformity.min():.3f} - {pareto_uniformity.max():.3f}")
        
        # Correlation analysis
        coverage_uniformity_corr = pearsonr(feasible_coverage, feasible_uniformity)[0]
        print(f"   • Coverage-Uniformity Correlation: {coverage_uniformity_corr:.3f}")
        
    else:
        print(f"   ⚠️ Insufficient feasible points for trade-off analysis ({len(feasible_coverage)} points)")
    
    # ====================
    # 7. NEXT SUGGESTIONS QUALITY ANALYSIS
    # ====================
    
    print(f"\n🎯 Next Suggestions Quality:")
    
    if next_experiments is not None and len(next_experiments) > 0:
        print(f"   • Generated Suggestions: {len(next_experiments)}")
        
        # Analyze suggestion diversity
        suggestion_values = next_experiments[param_names].values
        
        for i, param in enumerate(param_names):
            values = suggestion_values[:, i]
            param_range = param_ranges[param]
            
            diversity = (values.max() - values.min()) / (param_range[1] - param_range[0])
            print(f"   • {param} Diversity: {diversity:.1%} of full range")
        
        # Distance from training data
        from scipy.spatial.distance import cdist
        normalized_suggestions = (suggestion_values - np.array([param_ranges[p][0] for p in param_names])) / np.array([param_ranges[p][1] - param_ranges[p][0] for p in param_names])
        normalized_training = (X_features - np.array([param_ranges[p][0] for p in param_names])) / np.array([param_ranges[p][1] - param_ranges[p][0] for p in param_names])
        
        distances = cdist(normalized_suggestions, normalized_training)
        min_distances = distances.min(axis=1)
        
        print(f"   • Min Distance to Training: {min_distances.min():.3f}")
        print(f"   • Max Distance to Training: {min_distances.max():.3f}")
        print(f"   • Avg Distance to Training: {min_distances.mean():.3f}")
        
        exploration_suggestions = (min_distances > 0.2).sum()
        exploitation_suggestions = (min_distances <= 0.1).sum()
        print(f"   • Exploration Suggestions: {exploration_suggestions}/{len(next_experiments)}")
        print(f"   • Exploitation Suggestions: {exploitation_suggestions}/{len(next_experiments)}")
        
    else:
        print(f"   ⚠️ No suggestions available for analysis")
    
    # ====================
    # 8. MODEL CONFIDENCE ASSESSMENT
    # ====================
    
    print(f"\n🎯 Model Confidence Assessment:")
    
    # Calculate overall model quality metrics
    data_efficiency = phase_purity_success / len(y_train_raw)
    parameter_coverage = np.mean([(X_train[p].max() - X_train[p].min()) / (param_ranges[p][1] - param_ranges[p][0]) for p in param_names])
    
    print(f"   • Data Efficiency (feasible rate): {data_efficiency:.1%}")
    print(f"   • Parameter Space Coverage: {parameter_coverage:.1%}")
    print(f"   • Training Set Size: {len(X_train)} experiments")
    
    # Confidence level assessment
    if data_efficiency > 0.5 and parameter_coverage > 0.6 and len(X_train) >= 10:
        confidence = "HIGH"
    elif data_efficiency > 0.3 and parameter_coverage > 0.4 and len(X_train) >= 6:
        confidence = "MEDIUM"
    else:
        confidence = "LOW"
    
    print(f"   • Overall Model Confidence: {confidence}")
    
    # ====================
    # 9. RECOMMENDATIONS
    # ====================
    
    print(f"\n💡 RECOMMENDATIONS:")
    
    if phase_purity_rate < 0.5:
        print(f"   🔴 Low constraint satisfaction ({phase_purity_rate:.1%})")
        print(f"      → Focus on feasible region exploration")
    
    if parameter_coverage < 0.5:
        print(f"   🟡 Limited parameter space coverage ({parameter_coverage:.1%})")
        print(f"      → Consider space-filling designs")
    
    if len(X_train) < 15:
        print(f"   🟡 Small training set ({len(X_train)} samples)")
        print(f"      → Collect more data for robust modeling")
    
    if confidence == "HIGH":
        print(f"   🟢 Model ready for optimization")
        print(f"      → Trust suggestions and focus on exploitation")
    
    return {
        'cv_results': cv_results,
        'constraint_satisfaction_rate': phase_purity_rate,
        'parameter_coverage': parameter_coverage,
        'model_confidence': confidence,
        'pareto_points': pareto_points if 'pareto_points' in locals() else 0
    }

# Run the comprehensive analysis
analysis_results = comprehensive_model_analysis()

🔬 COMPREHENSIVE MODEL PERFORMANCE ANALYSIS


NameError: name 'X_train' is not defined